In [3]:
import sys
sys.path.append("..")

from src.data import load_data

df_model, df_enrolled = load_data("../data/data.csv")
print(df_model.shape, df_enrolled.shape)

(3630, 38) (794, 37)


In [4]:
second_sem_cols = [col for col in df_model.columns if col.startswith("curricular_units_2nd_sem")]

X = df_model.drop(columns=second_sem_cols + ["target", "dropout"])
y = df_model["dropout"]

print(second_sem_cols)
print(X.shape)

['curricular_units_2nd_sem_credited', 'curricular_units_2nd_sem_enrolled', 'curricular_units_2nd_sem_evaluations', 'curricular_units_2nd_sem_approved', 'curricular_units_2nd_sem_grade', 'curricular_units_2nd_sem_without_evaluations']
(3630, 30)


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42,
)

print(X_train.shape, X_test.shape)
print(y_train.mean().round(3), y_test.mean().round(3))

(2904, 30) (726, 30)
0.392 0.391


In [9]:
X_train.nunique().sort_values()

daytime_evening_attendance                        2
displaced                                         2
debtor                                            2
educational_special_needs                         2
international                                     2
scholarship_holder                                2
gender                                            2
tuition_fees_up_to_date                           2
application_order                                 6
marital_status                                    6
inflation_rate                                    9
unemployment_rate                                10
gdp                                              10
curricular_units_1st_sem_without_evaluations     11
previous_qualification                           16
course                                           17
application_mode                                 17
nationality                                      18
curricular_units_1st_sem_credited                21
curricular_u

In [11]:
categorical_cols = [
    "marital_status", 
    "nationality", 
    "application_mode", 
    "course", 
    "previous_qualification", 
    "mother_s_qualification", 
    "father_s_qualification", 
    "mother_s_occupation",
    "father_s_occupation"
    ]


binary_cols = [
    "gender", 
    "international", 
    "displaced", 
    "educational_special_needs", 
    "daytime_evening_attendance", 
    "debtor", 
    "tuition_fees_up_to_date", 
    "scholarship_holder"
    ]


numerical_cols = [
    "age_at_enrollment", 
    "application_order", 
    "previous_qualification_grade", 
    "admission_grade", 
    "curricular_units_1st_sem_credited", 
    "curricular_units_1st_sem_enrolled", 
    "curricular_units_1st_sem_evaluations",
    "curricular_units_1st_sem_approved", 
    "curricular_units_1st_sem_grade", 
    "curricular_units_1st_sem_without_evaluations", 
    "unemployment_rate", 
    "inflation_rate", 
    "gdp"
    ]


assert len(categorical_cols) + len(binary_cols) + len(numerical_cols) == X.shape[1]
assert set(categorical_cols + binary_cols + numerical_cols) == set(X.columns)

In [12]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("num", StandardScaler(), numerical_cols),
        ("bin", "passthrough", binary_cols),
    ]
)

X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

In [13]:
print(X_train_processed.shape, X_test_processed.shape)

(2904, 221) (726, 221)
